# Reproducing Silkuniene et al. (2025) from dandiset 001276

**Paper.** Silkuniene G, Silkunas M, Pakhomova ON, Semenov I, Pakhomov AG (2025). MHz compression of
pulse packets facilitates remote focusing of electroporation. *Bioelectrochemistry* 166:109016.
[doi:10.1016/j.bioelechem.2025.109016](https://doi.org/10.1016/j.bioelechem.2025.109016). The claims
file was written from the accepted author manuscript in PubMed Central (PMC12331426), with its
Methods and figure legends; the paper has no supplement and links no code.

**Dandiset.** [DANDI:001276](https://dandiarchive.org/dandiset/001276/0.250701.1609), version
0.250701.1609. The paper cites no version of the dandiset (its data statement says "Data will be made
available on request"). This is the only published version, and the most recent; the draft has the
same 108 assets and size, so there is nothing to choose between.

**What the dandiset holds.** One stitched whole-well image per file: YO-PRO-1 (YP) after exposure, or
Hoechst before it, for 54 wells of CT26 and CHO monolayers. The wells are the burst-number series of
Fig. 7 (0.2 MHz, 600 ns pulses, 1, 2, 4 or 6 bursts (x4)), and in addition CANCAN and control wells at
a packet rate of 1 Hz, 2 bursts (x4). There are no 1 kHz or 10 kHz wells, no raised electrodes, no
propidium images, no 300 or 900 ns pulses and no thermometry, so the claims on those are blocked. Three
other dandisets of the same lab cite this paper and, by their titles and descriptions, hold those series:
DANDI:001047 (1 Hz against 0.2 MHz, with propidium), DANDI:001277 (300, 600 and 900 ns pulses) and
DANDI:001284 (electrodes raised 0, 1 or 2 mm, with propidium). The claims file names no data outside this
dandiset, so they are not used here; nothing in 001276 points to them.

| Claim | Tested here |
|---|---|
| fig7be-center-linear-with-bursts | tested |
| fig7b-ct26-saturation | tested |
| fig7cf-fwhm-with-bursts | tested |
| fig7-sharpest-peak-one-burst | tested |
| fig7-offtarget-40-50-fold | tested |
| fig7-cho-replicates-ct26 | tested |
| fig4b-40-fold-at-02mhz | tested |
| fig3a-cancan-center-enhanced | tested on the rates present (1 Hz, 0.2 MHz, control) |
| fig3bc-lorentzian-profile | tested on the rates present (1 Hz, 0.2 MHz) |
| fig3bc-offtarget-deviation-falls-with-rate | tested (1 Hz against 0.2 MHz) |
| fig4b-center-no-rate-effect | tested on the rates present (0.2 MHz against 1 Hz) |
| fig4b-near-electrode-rate-decline | tested on the rates present (1 Hz, 0.2 MHz) |
| fig4c-fwhm-decreases-with-rate | tested on the rates present (1 Hz, 0.2 MHz) |
| fig4c-log-dependence | blocked: no 1 kHz or 10 kHz wells (a rate series is in DANDI:001047) |
| fig5-raised-electrodes-weaker-center | blocked: no raised-electrode wells (in DANDI:001284) |
| fig5b-pr-gaussian-yp-lorentzian | blocked: no propidium images (in DANDI:001284); the YP half is shown |
| fig5-yp-spot-slightly-larger-than-pr | blocked: no propidium images (in DANDI:001284) |
| fig5c-irreversible-center | blocked: no post-exposure Hoechst images (in DANDI:001284), and no Hoechst background in the paper; the YP part and pre-exposure Hoechst are shown |
| fig6b-900ns-peak-not-significant | blocked: no 300 or 900 ns wells (in DANDI:001277) |
| fig6c-normalized-decline-similar | blocked: no 300 or 900 ns wells (in DANDI:001277) |
| fig6d-fwhm-sharper-longer-pulses | blocked: no 300 or 900 ns wells (in DANDI:001277) |
| fig2-heating-below-44c | blocked: no thermochromic film images |
| fig1e-field-simulation | blocked: the field model is not public |

No random numbers are used anywhere in this notebook.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats

import cancan

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = cancan.list_assets()
meta = cancan.load_metadata(assets)
meta["date"] = meta.subject.str.split("_").str[1]
meta["plate"] = meta.subject.str.split("_").str[0]
print(f"dandiset {cancan.DANDISET_ID} version {cancan.DANDISET_VERSION}: {len(assets)} assets, "
      f"{meta.size_mb.sum() / 1e3:.1f} GB; metadata read in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files by channel and phase:")
print(meta.groupby(["channel", "phase"]).size().to_string())
print("\nImage shapes, dtypes and chunking:")
print(meta.groupby(["shape", "dtype", "chunks"]).size().to_string())
wells = meta.groupby("subject").agg(
    cell_line=("cell_line", "first"), species=("species", "first"), protocol=("protocol", "first"),
    rate=("packet_rate", "first"), bursts=("bursts", "first"), code=("protocol_code", "first"),
    date=("date", "first"), plate=("plate", "first"),
    files=("path", "size"), channels=("channel", lambda c: "+".join(sorted(c))),
    consistent=("bursts", "nunique"))
assert (wells.consistent == 1).all()
print(f"\n{len(wells)} wells (subjects)")
print("\nWells by cell line, protocol, packet rate and number of bursts (x4):")
print(wells.groupby(["cell_line", "protocol", "rate", "bursts"]).size().to_string())
print("\nWells whose files are not one DAPI (pre) and one FITC (post):")
print(wells[wells.channels != "DAPI+FITC"][["cell_line", "protocol", "rate", "bursts", "channels"]].to_string())
print("\nCell line against species:")
print(wells.groupby(["cell_line", "species"]).size().to_string())
print("\nProtocol code against protocol and rate:", wells.groupby(["code", "protocol", "rate"]).size().to_dict())
print("\nWells per experiment date (from the subject id) and cell line:")
print(pd.crosstab(wells.date, wells.cell_line).to_string())
print("\nsession_start_time values (dates):", sorted(meta.session_start.str[:10].unique()))
print("Inter-electrode distance stated in every description:",
      sorted(set(meta.description.str.extract(r"inter-electrode distance of ([\d.]+ mm)")[0])))
print("\nOME-XML at the end of each description: pixel size (um), exposure (ms), objective and camera, by channel:")
print(meta.groupby(["channel", "um_per_px_x", "um_per_px_y", "exposure_ms", "objective_mag", "detector"], dropna=False).size().to_string())
print("Files without OME-XML:", meta[meta.um_per_px_x.isna()].path.tolist())
pos = meta.groupby(["subject", "channel"])[["stage_x_um", "stage_y_um"]].first().unstack("channel")
both = pos.notna().all(axis=1)
same = (pos[("stage_x_um", "DAPI")] == pos[("stage_x_um", "FITC")]) & (pos[("stage_y_um", "DAPI")] == pos[("stage_y_um", "FITC")])
print(f"Wells whose Hoechst and YP images have the same stage position: {int(same[both].sum())} of {int(both.sum())} with both recorded")


The dandiset holds 54 wells (its "subjects"), each a well of a 6-well plate. Cell line and species agree
(CT26 with *Mus musculus*, CHO with *Cricetulus griseus*). The protocol, packet rate and number of bursts
are only in the free-text description, where code 011 is 0.2 MHz and code 013 is 1 Hz. Compared with the
paper's Fig. 7 (CT26 n = 3 to 5, CHO n = 4), the dandiset has 3, 6, 3 and 3 CT26 wells and 5, 8, 7 and 8
CHO wells for 1, 2, 4 and 6 bursts (x4), more CHO wells than the paper reports, with no record of which
wells the paper used. It also has CANCAN and control wells at 1 Hz, 2 bursts (x4): CT26 (2 and 4 wells,
the paper has n = 3 and n = 5 controls for Fig. 3 and 4) and CHO (3 and 2 wells, not in the paper). Every
`session_start_time` is 2024-12-12, the conversion date; the experiment dates are only in the subject ids
(June to September 2024). One well has no Hoechst file and one has two (below). The stated
"inter-electrode distance of 10.0 mm" fits the paper's geometry only as the edge-to-edge distance along a
side (12.8 mm between centers less 2.8 mm); the electrode fits below give 12.8 mm between centers. The OME-XML appended to each description gives a pixel size of 1.625 um for every file that has it,
the 4x objective and ORCA-Flash4.0 camera of Methods 2.4, and one exposure per channel (200 ms for YP,
600 ms for Hoechst), so YP intensities are comparable across wells. The Hoechst and YP images of a well
have the same stage position, so they share coordinates.


In [ ]:
odd = meta[(meta.channel == "DAPI") & meta.subject.isin(["P1_20240821_A1", "P1_20240821_A2"])]
rows = []
for _, r in odd.iterrows():
    planes = r["shape"][1] if len(r["shape"]) == 4 else 1
    for plane in range(planes):
        b = cancan.load_binned(r.url, plane=plane)
        im = b["image"]
        rows.append({"file": r.path.split("/")[-1], "shape": r["shape"], "size_mb": round(r.size_mb, 1),
                     "plane": plane, "fraction_of_area_imaged": round(float(np.mean(b["inside"] > 0)), 3),
                     "median": round(float(np.nanmedian(im)), 1), "p99": round(float(np.nanpercentile(im, 99)), 1)})
        if r.path.endswith("fkppt9_image.nwb") and plane == 0:
            frag = im
        if r.path.endswith("10gndju_image.nwb"):
            full = im
odd_planes = pd.DataFrame(rows)
print(odd_planes.to_string(index=False))
m = np.isfinite(frag) & np.isfinite(full)
print(f"\nThe 12.5 MB file of P1_20240821_A2 covers {m.sum()} bins; there its plane 0 correlates with the full "
      f"Hoechst image of the same well at r = {np.corrcoef(frag[m], full[m])[0, 1]:.3f}")

Two Hoechst files have a second image plane (shape (1, 2, H, W)) that the description does not mention;
its values (median about 50) are near the camera offset, so it looks like an empty channel. Well
P1_20240821_A2 has a second Hoechst file that covers 1% of the well and matches the same region of its
full Hoechst image: a fragment of the same acquisition deposited twice. The Hoechst images are used only for the
check of fig5c-irreversible-center below; the empty planes give the camera offset there.

Next, every YP image is streamed and reduced to 10 x 10 pixel bins (16.25 um) as it loads.

In [ ]:
yp_files = meta[meta.channel == "FITC"].set_index("subject")
t0 = time.time()
images, sat_maps, load_info = {}, {}, []
for subject, r in yp_files.iterrows():
    b = cancan.load_binned(r.url)
    images[subject] = b["image"]
    sat_maps[subject] = b["saturated"]
    im = b["image"]
    load_info.append({"subject": subject, "MB": round(b["bytes"] / 1e6), "chunks": b["n_chunks"],
                      "imaged_fraction": round(float(np.mean(b["inside"] > 0)), 3),
                      "bin_min": float(np.nanmin(im)), "bin_median": float(np.nanmedian(im)),
                      "bin_max": float(np.nanmax(im)),
                      "pixels_near_full_scale": float(np.nansum(b["saturated"] * b["inside"]) * cancan.BIN ** 2)})
load_info = pd.DataFrame(load_info).set_index("subject")
print(f"{len(images)} YP images, {load_info.MB.sum() / 1e3:.1f} GB streamed in {(time.time() - t0) / 60:.1f} min")
print("\nRanges of the 10 x 10 bin means over all images (a.u., uint16 camera counts):")
print(load_info[["imaged_fraction", "bin_min", "bin_median", "bin_max"]].describe().round(1).to_string())
print(f"\nPixels at or above {cancan.NEAR_FULL_SCALE} (uint16 maximum 65535): at most {load_info.pixels_near_full_scale.max():.0f} "
      f"in one image of {19190 ** 2} pixels")


### Locating the electrode array

The files do not give the electrode positions. The pixel size is 1.625 um, from the OME-XML of the
descriptions (above), which is also 6.5 um camera pixels behind the 4x objective of Methods 2.4. The
electrodes are found as dark footprints ("Footprints of 4 electrodes are identified by the lack of YP
signal underneath", Fig. 3A legend): a square of the paper's size (12.8 mm between electrode centers) is
placed where its four corners are darkest against their surroundings. The fit is repeated with the side
left free, which checks the pixel size against the electrode geometry. A well is used only when the two
fits agree (centers within 0.5 mm, free side within 4% of 12.8 mm); this rule was set here, as the paper
places ROIs by eye.

In [ ]:
fits = []
squares = {}
for subject, im in images.items():
    sq = cancan.square_fit(im)
    sqf = cancan.square_fit(im, free_side=True)
    squares[subject] = sq
    fits.append({"subject": subject, "cx_mm": sq["cx"] * cancan.UM_PER_BIN / 1000,
                 "cy_mm": sq["cy"] * cancan.UM_PER_BIN / 1000, "angle_deg": sq["theta_deg"],
                 "corner_contrast_min": float(np.min(sq["corner_scores"])),
                 "free_side_mm": sqf["side_mm"],
                 "center_shift_mm": float(np.hypot(sq["cx"] - sqf["cx"], sq["cy"] - sqf["cy"]) * cancan.UM_PER_BIN / 1000)})
fits = pd.DataFrame(fits).set_index("subject").join(wells[["cell_line", "protocol", "rate", "bursts"]])
fits["used"] = (fits.center_shift_mm < 0.5) & ((fits.free_side_mm - cancan.SIDE_MM).abs() < 0.04 * cancan.SIDE_MM)
print(fits.round(3).to_string())
used = fits[fits.used]
print(f"\nWells used: {fits.used.sum()} of {len(fits)}. Left out:")
print(fits[~fits.used][["cell_line", "protocol", "rate", "bursts", "free_side_mm", "center_shift_mm"]].round(2).to_string())
print(f"\nFree side over the wells used: median {used.free_side_mm.median():.2f} mm, range "
      f"{used.free_side_mm.min():.2f} to {used.free_side_mm.max():.2f} mm (paper geometry 12.80 mm at 1.625 um per pixel)")

In [ ]:
fig, axs = plt.subplots(6, 9, figsize=(27, 18))
th = np.linspace(0, 2 * np.pi, 60)
r_fp = 1400 / cancan.UM_PER_BIN
for ax, (subject, im) in zip(axs.ravel(), images.items()):
    s = im[::4, ::4]
    ax.imshow(s, cmap="gray", vmin=np.nanpercentile(s, 1), vmax=np.nanpercentile(s, 99.7))
    sq = squares[subject]
    c = np.vstack([sq["corners"], sq["corners"][:1]]) / 4
    ax.plot(c[:, 0], c[:, 1], "r-", lw=0.6)
    for k in sq["corners"] / 4:
        ax.plot(k[0] + r_fp / 4 * np.cos(th), k[1] + r_fp / 4 * np.sin(th), "c-", lw=0.5)
    f = fits.loc[subject]
    ax.set_title(f"{subject} {f.cell_line} {f.protocol} {f.rate} {f.bursts}b" + ("" if f.used else " (left out)"), fontsize=7)
    ax.axis("off")
plt.suptitle("YP images (intensity between the 1st and 99.7th percentile of each image); red: fitted electrode square; cyan: 2.8 mm footprints")
plt.tight_layout()
plt.show()

In [ ]:
fits["date"], fits["plate"] = wells.date, wells.plate
alt_squares, rows = {}, []
for subject in fits[~fits.used].index:
    f = fits.loc[subject]
    same = fits[fits.used & (fits.date == f.date) & (fits.plate == f.plate)]
    sq = cancan.square_fit(images[subject], angle_deg=same.angle_deg.median())
    alt_squares[subject] = sq
    rows.append({"subject": subject, "wells_same_plate_day": len(same), "their_angles_deg": f"{same.angle_deg.min():.1f} to {same.angle_deg.max():.1f}",
                 "angle_free_fit_deg": f.angle_deg, "angle_used_deg": sq["theta_deg"],
                 "shift_from_free_fit_mm": float(np.hypot(sq["cx"] - squares[subject]["cx"], sq["cy"] - squares[subject]["cy"]) * cancan.UM_PER_BIN / 1000),
                 "corner_contrast_min": float(np.min(sq["corner_scores"])), "corner_contrast_mean": float(np.mean(sq["corner_scores"]))})
print("Wells left out, refitted with the angle fixed at the median of the wells of the same plate and day:")
print(pd.DataFrame(rows).set_index("subject").round(3).to_string())
print(f"Corner contrast over the wells used: min {fits[fits.used].corner_contrast_min.min():.3f}, median {fits[fits.used].corner_contrast_min.median():.3f}")
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
th = np.linspace(0, 2 * np.pi, 60)
for ax, (subject, sq) in zip(axs, alt_squares.items()):
    s = images[subject][::2, ::2]
    ax.imshow(s, cmap="gray", vmin=np.nanpercentile(s, 1), vmax=np.nanpercentile(s, 99.5))
    c = np.vstack([sq["corners"], sq["corners"][:1]]) / 2
    ax.plot(c[:, 0], c[:, 1], "r-", lw=0.8)
    for k in sq["corners"] / 2:
        ax.plot(k[0] + r_fp / 2 * np.cos(th), k[1] + r_fp / 2 * np.sin(th), "c-", lw=0.6)
    ax.set_title(f"{subject} (left out), angle fixed", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


The square of fixed size sits on the visible footprints in the wells used. In the three wells left out
(one CT26 well with 2 bursts (x4) and two CHO wells with 1 burst (x4)) the footprints are too faint, or
lie partly outside the image, for the two fits to agree, and the free fits take angles (13 to 45 degrees)
far from those of the other wells. Wells imaged on the same plate and day share an angle to within a few
degrees (table above), so as a variant each left-out well is refitted with the angle fixed at the median
of its plate and day. Faint footprint rims lie under some of its corners (figure), but its smallest
corner contrast (0.013 to 0.045) is below the median of the wells used (0.333), so the placement is less
certain than for the wells used. The claims are
judged on the wells used, and the results with these three wells added are shown as a variant where they
enter. Over the wells used the free side is close to
12.8 mm, which is consistent with 1.625 um per pixel and with the paper's electrode geometry, though it
does not prove either alone.

The next check asks whether the electrode positions and the YP image share coordinates in the way the
paper describes: in CANCAN wells the YP peak should lie at the array center, and in control wells, where
the field is unipolar, YP should not peak there. The peak is the maximum of the image smoothed over 1 mm,
searched within 4 mm of the fitted center.

In [ ]:
from scipy import ndimage
rows = []
for subject in used.index:
    im = np.nan_to_num(images[subject], nan=np.nanmedian(images[subject]))
    sm = ndimage.uniform_filter(im, size=int(1000 / cancan.UM_PER_BIN))
    sq = squares[subject]
    yy, xx = np.mgrid[0:im.shape[0], 0:im.shape[1]]
    dist = np.hypot(xx - sq["cx"], yy - sq["cy"]) * cancan.UM_PER_BIN / 1000
    sm[dist > 4] = -np.inf
    iy, ix = np.unravel_index(np.argmax(sm), sm.shape)
    rows.append({"subject": subject, "offset_mm": dist[iy, ix],
                 "peak_over_4mm_ring": sm[iy, ix] / np.mean(im[(dist > 3.8) & (dist < 4.2)])})
peaks = pd.DataFrame(rows).set_index("subject").join(fits[["cell_line", "protocol", "rate", "bursts"]])
print("Offset of the YP peak from the fitted array center (mm), and peak over the mean at 4 mm:")
print(peaks.groupby(["cell_line", "protocol", "rate"])[["offset_mm", "peak_over_4mm_ring"]].describe().round(2)
      [[("offset_mm", "count"), ("offset_mm", "50%"), ("offset_mm", "max"), ("peak_over_4mm_ring", "50%")]].to_string())

In the 0.2 MHz CANCAN wells the YP peak lies 0.31 mm (CT26) and 0.36 mm (CHO) from the fitted center
(medians; at most 1.47 mm), and is 3.6 to 4.1 times the mean at 4 mm. In control wells the brightest
point of the search area lies 3.2 to 3.9 mm out (medians), at its edge. This is consistent with the
footprint fits marking the array center the paper measures from, and the CANCAN peaks being centered. The images are single time points, so there is no clock to check.

### Profiles

For every well (the left-out wells with the variant placement, kept apart): the background (3 ROIs in
each footprint, median over the four footprints, with each footprint kept for the readings below), and
the ROI means along the two diagonals and the two lines through the center parallel to the sides, at
ROI spacings of 1 and 0.5 mm out to 7 mm (the readings of the claims file), background subtracted and
the two lines averaged at each signed distance (Methods 2.5). The share of camera pixels at full scale in
each diagonal ROI checks for saturation.

In [ ]:
def well_profiles(subject, sq, im):
    bg = cancan.background(im, sq)
    out = []
    for spacing in (1.0, 0.5):
        for kind in ("diagonal", "sides"):
            p = cancan.profile(im, sq, cancan.distances(spacing), kind)
            for x, v in p.groupby("x_mm").raw.mean().items():
                out.append({"subject": subject, "spacing": spacing, "kind": kind, "x": x, "raw": v, "yp": v - bg["background"]})
    return bg, out

bg_rows, prof_rows, alt_rows = [], [], []
for subject in fits.index:
    sq = squares[subject] if fits.loc[subject, "used"] else alt_squares[subject]
    bg, out = well_profiles(subject, sq, images[subject])
    bg_rows.append({"subject": subject, "background": bg["background"],
                    **{f"footprint_{i}": v for i, v in enumerate(bg["per_footprint"])}})
    (prof_rows if fits.loc[subject, "used"] else alt_rows).extend(out)
bgs = pd.DataFrame(bg_rows).set_index("subject")
cols = ["cell_line", "protocol", "rate", "bursts", "date"]
profiles = pd.DataFrame(prof_rows).merge(wells[cols], left_on="subject", right_index=True)
profiles_alt = pd.DataFrame(alt_rows).merge(wells[cols], left_on="subject", right_index=True)
fp = bgs.loc[used.index, [c for c in bgs if c.startswith("footprint")]]
spread = (fp.max(axis=1) - fp.min(axis=1)) / bgs.loc[used.index].background
print("Background (a.u.), wells used: range over wells", round(bgs.loc[used.index].background.min(), 1), "to", round(bgs.loc[used.index].background.max(), 1))
print("Footprints outside the image:", int(fp.isna().sum().sum()))
print("Spread between footprints, (max - min) / background: median", round(spread.median(), 3), "max", round(spread.max(), 3),
      "in", spread.idxmax(), "; in a.u., median", round((fp.max(axis=1) - fp.min(axis=1)).median(), 1),
      "range", round((fp.max(axis=1) - fp.min(axis=1)).min(), 1), "to", round((fp.max(axis=1) - fp.min(axis=1)).max(), 1))
print("Profile ROIs outside the imaged area:", int(profiles.yp.isna().sum()))
print("Background-subtracted YP over all profile ROIs: min", round(profiles.yp.min(), 1), "max", round(profiles.yp.max(), 1))
sat_rows = []
for subject in used.index:
    p = cancan.profile(sat_maps[subject], squares[subject], cancan.distances(1.0), "diagonal")
    sat_rows.append(p.assign(subject=subject))
sat = pd.concat(sat_rows).merge(wells[cols], left_on="subject", right_index=True)
print("\nFraction of pixels at or above", cancan.NEAR_FULL_SCALE, "in the diagonal ROIs (1 mm spacing), largest over wells, by group and |distance|:")
print(sat.assign(d=sat.x_mm.abs()).pivot_table(index=["cell_line", "protocol", "rate", "bursts"], columns="d", values="raw", aggfunc="max").map(lambda v: f"{v:.1e}").to_string())


In [ ]:
FIT_RANGES = (4, 5, 6)
def measures(prof):
    rows = []
    for (subject, spacing), d in prof[prof.kind == "diagonal"].groupby(["subject", "spacing"]):
        s = d.set_index("x").yp
        row = {"subject": subject, "spacing": spacing, "center": s.loc[0.0],
               "yp6": s[s.index.map(abs) == 6].mean(), "yp7": s[s.index.map(abs) == 7].mean()}
        for R in FIT_RANGES:
            f = cancan.fit_lorentz(s.index.values, s.values, R)
            row[f"fwhm{R}"], row[f"R{R}"], row[f"bound{R}"] = f["b"], f["R"], f["b"] >= 99.9
        rows.append(row)
    out = pd.DataFrame(rows).merge(wells[cols], left_on="subject", right_index=True)
    out["ratio6"], out["ratio7"] = out.center / out.yp6, out.center / out.yp7
    return out

meas, meas_alt = measures(profiles), measures(profiles_alt)
m1 = meas[meas.spacing == 1.0]
print("Per-well measures, ROI spacing 1 mm (YP in a.u. after background; FWHM in mm by fit range):")
print(m1.drop(columns=["spacing"]).round(2).to_string(index=False))
print("\nLorentzian fits whose FWHM ran to the 100 mm bound (no peak), by group:")
print(meas.groupby(["cell_line", "protocol", "rate", "spacing"])[[f"bound{R}" for R in FIT_RANGES]].sum().to_string())

The background differs between the four footprints of a well by a median of 15% (37.5 a.u.), up to
399.5 a.u. in one CHO well, which is as large as YP near the electrodes in the CT26 0.2 MHz wells. At most
5.4e-4 of the pixels of any diagonal ROI are at full scale, so saturation does not shape the profiles.

Background readings. Methods 2.5 takes the background from 3 ROIs "within a footprint of an electrode",
one footprint, without saying which. The main analysis uses the median of the four footprints (an
assumption, `cancan.py`). Near the electrodes YP after background is small, so the ratio claims are also
computed with each single footprint as the background (named by its corner of the image), with the lowest
and the highest of the four, and over every way of picking one footprint per well.

In [ ]:
import itertools
raw_all = pd.concat([profiles, profiles_alt])
raw_d1 = raw_all[(raw_all.kind == "diagonal") & (raw_all.spacing == 1.0)]
near = raw_d1.assign(d=raw_d1.x.abs()).groupby(["subject", "d"]).raw.mean().unstack("d")[[0.0, 6.0, 7.0]]
near.columns = ["c_raw", "r6_raw", "r7_raw"]
FP_COLS = [f"footprint_{i}" for i in range(4)]
BG_READINGS = {"median of 4 (main)": bgs.background,
               **{f"footprint {name}": bgs[c] for c, name in zip(FP_COLS, ["upper left", "upper right", "lower right", "lower left"])},
               "lowest footprint": bgs[FP_COLS].min(axis=1), "highest footprint": bgs[FP_COLS].max(axis=1)}

def ratios_bg(subjects, bg):
    """Center over near-electrode YP, readings (i) of group means and (ii) mean of per-well ratios."""
    c = near.loc[subjects, "c_raw"].values - np.asarray(bg)
    out = {}
    for k in ("6", "7"):
        y = near.loc[subjects, f"r{k}_raw"].values - np.asarray(bg)
        out[f"{k}mm_i_group_means"], out[f"{k}mm_ii_mean_of_ratios"] = c.mean() / y.mean(), (c / y).mean()
    return out

def assignment_share(subjects, lo, hi):
    """Share of the 4^n ways of taking the background from one footprint per well that put each ratio in [lo, hi]."""
    fpv = bgs.loc[subjects, FP_COLS].values
    hits, n = None, 0
    for combo in itertools.product(range(4), repeat=len(subjects)):
        r = ratios_bg(subjects, fpv[np.arange(len(subjects)), combo])
        ok = pd.Series({k: lo <= v <= hi for k, v in r.items()})
        ok["any reading"] = ok.any()
        hits = ok.astype(int) if hits is None else hits + ok.astype(int)
        n += 1
    return hits / n, n


In [ ]:
grp = ["cell_line", "protocol", "rate", "bursts"]
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (cell, prot, rate) in zip(axs, [("CT26", "cancan", "0.2MHz"), ("CHO", "cancan", "0.2MHz"), ("CT26", "any", "1Hz")]):
    if prot == "any":
        sel = profiles[(profiles.cell_line == cell) & (profiles.rate == "1Hz")]
        keys = ["protocol"]
    else:
        sel = profiles[(profiles.cell_line == cell) & (profiles.protocol == prot) & (profiles.rate == rate)]
        keys = ["bursts"]
    sel = sel[(sel.kind == "diagonal") & (sel.spacing == 0.5)]
    for k, d in sel.groupby(keys):
        g = d.groupby("x").yp.agg(["mean", "sem"])
        ax.errorbar(g.index, g["mean"], g["sem"], label=f"{k[0]} (n={d.subject.nunique()})", capsize=2)
    ax.set_title(f"{cell} {rate} diagonal profiles, mean +/- s.e.")
    ax.set_xlabel("distance from center (mm)")
    ax.set_ylabel("YP, background subtracted (a.u.)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("Plotted range of mean YP (a.u.):", round(profiles.groupby(["subject", "kind", "spacing", "x"]).yp.mean().min(), 1),
      "to", round(profiles.groupby(["subject", "kind", "spacing", "x"]).yp.mean().max(), 1), "; all points inside the axes")

## fig7be-center-linear-with-bursts

> In both CT26 and CHO cells, YP uptake at the array center increases linearly with the number of
> bursts (slopes different from zero at p < 0.001), except for saturation at 6 bursts (x4) in CT26 cells.

Center YP is the ROI at 0 mm. Linear fit on burst number, the 6-burst point excluded for CT26; readings
(a) individual experiments and (b) group means.

In [ ]:
fig7 = m1[(m1.protocol == "cancan") & (m1.rate == "0.2MHz")]
def center_fits(cell, bursts):
    d = fig7[(fig7.cell_line == cell) & fig7.bursts.isin(bursts)]
    g = d.groupby("bursts").center.mean()
    return {"a_individual": cancan.slope_test(d.bursts, d.center), "b_group_means": cancan.slope_test(g.index, g.values)}
print("Center YP by burst number (a.u.), mean, s.e., n, and per experiment date:")
print(fig7.groupby(["cell_line", "bursts"]).center.agg(["mean", "sem", "count"]).round(1).to_string())
print(fig7.pivot_table(index=["cell_line", "date"], columns="bursts", values="center", aggfunc="mean").round(0).to_string())
lin = {}
for cell, b in [("CT26", [1, 2, 4]), ("CHO", [1, 2, 4, 6]), ("CT26 with 6 (variant)", [1, 2, 4, 6])]:
    for reading, r in center_fits(cell.split()[0], b).items():
        lin[(cell, reading)] = r
        print(f"{cell:24s} bursts {b} {reading:14s} slope {r['slope']:8.1f} a.u./burst  s.e. {r['se']:6.1f}  "
              f"r {r['r']:.3f}  n {r['n']:2d}  p {r['p']:.2g}")
print("\nVariant: the three wells left out added, placed with the angle of their plate and day:")
fig7_alt = pd.concat([fig7, meas_alt[(meas_alt.spacing == 1.0) & (meas_alt.protocol == "cancan") & (meas_alt.rate == "0.2MHz")]])
for cell, b in [("CT26", [1, 2, 4]), ("CHO", [1, 2, 4, 6])]:
    d = fig7_alt[(fig7_alt.cell_line == cell) & fig7_alt.bursts.isin(b)]
    g = d.groupby("bursts").center.mean()
    ra, rb = cancan.slope_test(d.bursts, d.center), cancan.slope_test(g.index, g.values)
    print(f"{cell:5s} bursts {b} wells {ra['n']:2d}: slope {ra['slope']:.1f} a.u./burst, (a) p {ra['p']:.2g}, (b) p {rb['p']:.2g}")
print("\nVariant: background readings (wells used):")
rows = []
for name, bg in BG_READINGS.items():
    for cell, b in [("CT26", [1, 2, 4]), ("CHO", [1, 2, 4, 6])]:
        d = fig7[(fig7.cell_line == cell) & fig7.bursts.isin(b)]
        c = near.loc[d.subject, "c_raw"].values - bg.loc[d.subject].values
        g = pd.Series(c).groupby(d.bursts.values).mean()
        rows.append({"background": name, "cell_line": cell, "slope": cancan.slope_test(d.bursts, c)["slope"],
                     "p_a_individual": cancan.slope_test(d.bursts, c)["p"], "p_b_group_means": cancan.slope_test(g.index, g.values)["p"]})
print(pd.DataFrame(rows).pivot(index="background", columns="cell_line").to_string(float_format=lambda v: f"{v:.3g}"))


**Result: partially reproduced.** Center YP rises with burst number in both cell lines. The CHO slope
over all four burst numbers is positive with p < 0.001 under reading (a), individual experiments
(p = 2.2e-10). The CT26 slope over 1, 2 and 4 bursts (x4) is positive but its p is 0.0013 under reading
(a) and 0.024 under reading (b), so p < 0.001 holds in CHO only. CT26 has 11 wells here against the
paper's 3 to 5 per group, from three experiment dates. Under every background reading the CT26 p stays
between 0.0011 and 0.0015 (reading (a)). With the three left-out wells added the CT26 p is 0.00098 under
reading (a), just below 0.001, and 0.059 under reading (b), which would make the claim reproduced under
reading (a). The outcome therefore rests on leaving out well P1_20240627_B1, whose footprints the fit
could not find; this is the assumption adopted, since its variant placement is less certain.

## fig7b-ct26-saturation

> In CT26 cells the central YP uptake saturates at 6 bursts (x4), falling below the linear increase seen
> from 1 to 4 bursts, while CHO cells show no saturation.

The 6-burst mean against the 95% confidence band of the linear fit through 1, 2 and 4 bursts,
extrapolated to 6; readings (a) and (b) as above.

In [ ]:
sat = []
for cell in ("CT26", "CHO"):
    d = fig7[(fig7.cell_line == cell) & fig7.bursts.isin([1, 2, 4])]
    g = d.groupby("bursts").center.mean()
    obs6 = fig7[(fig7.cell_line == cell) & (fig7.bursts == 6)].center
    for reading, (x, y) in {"a_individual": (d.bursts, d.center), "b_group_means": (g.index, g.values)}.items():
        band = cancan.fit_band(x, y, [6]).iloc[0]
        sat.append({"cell_line": cell, "reading": reading, "fit_at_6": band.fit, "band_lo": band.lo, "band_hi": band.hi,
                    "observed_6_mean": obs6.mean(), "observed_6_sem": obs6.sem(), "n6": len(obs6),
                    "below_band": obs6.mean() < band.lo, "below_line": obs6.mean() < band.fit,
                    "within_band": band.lo <= obs6.mean() <= band.hi})
print(pd.DataFrame(sat).round(1).to_string(index=False))

**Result: partially reproduced.** The CT26 6-burst mean (4827 a.u.) lies below the line fitted through
1, 2 and 4 bursts (x4), extrapolated to 6 (6273), but inside its 95% band (lower edge 4234 under reading
(a), 4338 under reading (b)). The CHO 6-burst mean lies inside the band of the CHO fit under both readings,
as the claim expects.

## fig7cf-fwhm-with-bursts

> The FWHM of the central YP peak increases in proportion to the number of bursts in CT26 cells (slope
> p < 0.001) but not in CHO cells (p = 0.44).

Per-experiment Lorentzian FWHM; CT26 over 1, 2, 4 and 6 bursts (x4), CHO over 2, 4 and 6. Readings: fit
range (4, 5, 6 mm), ROI spacing (1, 0.5 mm), fit to experiments or to group means.

In [ ]:
fig7all = meas[(meas.protocol == "cancan") & (meas.rate == "0.2MHz")]
print("FWHM (mm) by burst number, mean, s.e., n (spacing 1 mm, fit range 5 mm):")
print(fig7all[fig7all.spacing == 1.0].groupby(["cell_line", "bursts"]).fwhm5.agg(["mean", "sem", "count"]).round(2).to_string())
rows = []
for spacing in (1.0, 0.5):
    for R in FIT_RANGES:
        for reading in ("individual", "group_means"):
            row = {"spacing": spacing, "fit_range": R, "reading": reading}
            for cell, b in (("CT26", [1, 2, 4, 6]), ("CHO", [2, 4, 6])):
                d = fig7all[(fig7all.cell_line == cell) & (fig7all.spacing == spacing) & fig7all.bursts.isin(b)]
                if reading == "individual":
                    r = cancan.slope_test(d.bursts, d[f"fwhm{R}"])
                else:
                    g = d.groupby("bursts")[f"fwhm{R}"].mean()
                    r = cancan.slope_test(g.index, g.values)
                row[f"{cell}_slope"], row[f"{cell}_p"] = r["slope"], r["p"]
            row["CT26_ok"] = row["CT26_slope"] > 0 and row["CT26_p"] < 0.001
            row["CHO_ok"] = row["CHO_p"] >= 0.05
            rows.append(row)
fw = pd.DataFrame(rows)
print(fw.round(4).to_string(index=False))
print("\nReadings where both parts hold:", int((fw.CT26_ok & fw.CHO_ok).sum()), "of", len(fw),
      "; CT26 part holds in", int(fw.CT26_ok.sum()), "; CHO part holds in", int(fw.CHO_ok.sum()))

**Result: reproduced.** Under the readings with 0.5 mm ROI spacing, a fit range of 5 or 6 mm and fits to
individual experiments, the CT26 slope is positive with p < 0.001 and the CHO slope has p >= 0.05
(0.064 and 0.090), as the criterion asks under one reading. The outcome depends on the reading: both
parts hold in 2 of the 12 readings. The CT26 part fails in every group-mean reading (p 0.008 to 0.029),
and the CHO part fails in four of the six individual-experiment readings (p 0.025 to 0.047), since the CHO
FWHM is larger at 6 bursts (2.65 mm) than at 2 and 4 (2.09 and 2.08 mm, fit range 5 mm, 1 mm spacing). The pass is fragile: the dataset has 23 CHO wells at 2, 4 and 6 bursts (x4) where the paper has 12, which
gives the small CHO slope (0.12 to 0.15 mm per burst) more power to reach p < 0.05 than in the paper.


## fig7-sharpest-peak-one-burst

> One burst (x4) in CT26 cells gives the sharpest peak of the study, FWHM 1.1 +/- 0.1 mm, 7.3% of the
> inter-electrode gap and 2.3% of the electrode array area.

Mean per-experiment FWHM of CT26 at 1 burst (x4); gap 15.3 mm; area readings (a) disk of diameter FWHM
over the 117 mm2 square of 15.3 mm diagonals, (b) disk of radius FWHM over the 164 mm2 square of the
electrode centers, (c) disk of diameter FWHM over 164 mm2. Fit range and ROI spacing under each reading.

In [ ]:
one = fig7all[(fig7all.cell_line == "CT26") & (fig7all.bursts == 1)]
print("Per-experiment FWHM, CT26, 1 burst (x4) (mm):")
print(one.pivot_table(index="subject", columns="spacing", values=[f"fwhm{R}" for R in FIT_RANGES]).round(3).to_string())
sq_diag = 15.3 ** 2 / 2
sq_centers = cancan.SIDE_MM ** 2
rows = []
for spacing in (1.0, 0.5):
    for R in FIT_RANGES:
        v = one[one.spacing == spacing][f"fwhm{R}"]
        f = v.mean()
        area = {"a": np.pi * (f / 2) ** 2 / sq_diag * 100, "b": np.pi * f ** 2 / sq_centers * 100,
                "c": np.pi * (f / 2) ** 2 / sq_centers * 100}
        rows.append({"spacing": spacing, "fit_range": R, "fwhm_mean": f, "fwhm_sem": v.sem(), "n": len(v),
                     "gap_pct": f / 15.3 * 100, **{f"area_{k}_pct": a for k, a in area.items()},
                     "fwhm_ok": abs(f - 1.1) <= 0.1 + 1e-12, "gap_ok": abs(f / 15.3 * 100 - 7.3) <= 0.73 + 1e-12,
                     "area_ok_any": any(abs(a - 2.3) <= 0.23 for a in area.values())})
sharp = pd.DataFrame(rows)
print(sharp.round(3).to_string(index=False))
print("\nSmallest group-mean FWHM among all 0.2 MHz groups (spacing 1 mm, fit range 5 mm):")
print(fig7all[fig7all.spacing == 1.0].groupby(["cell_line", "bursts"]).fwhm5.mean().sort_values().head(3).round(3).to_string())

**Result: reproduced, under the assumption of 0.5 mm ROI spacing.** CT26 with 1 burst (x4) gives the
smallest group-mean FWHM of the dataset. With 0.5 mm spacing the FWHM is 1.09 to 1.10 mm, 7.1 to 7.2% of
the gap, and the area under reading (b) is 2.29 to 2.33%, all within tolerance for every fit range. With
1 mm spacing the FWHM is 1.00 to 1.01 mm, still within tolerance, but the gap percentage (6.55 to 6.61%)
is at the lower edge of 6.57% (below it for the 4 mm fit range) and no area reading reaches 2.07%, which would make the claim
partially reproduced. The 0.5 mm spacing is adopted as the reading the paper better supports (ROIs
"connecting electrodes' footprints", Methods 2.5, and a peak 1 mm wide that a 1 mm grid samples with
three points). The three experiments give 0.92 to 1.31 mm (0.5 mm spacing).

## fig7-offtarget-40-50-fold

> With 1 burst (x4) in CT26 cells, YP uptake near the electrodes is 40 to 50 times smaller than at the
> remote target in the center.

Ratio of center YP to YP near electrodes, in 36 to 55. Readings: near electrodes at (a) 6 or (b) 7 mm;
ratio (i) of group means or (ii) mean of per-experiment ratios; group (A) 1 burst (x4) or (B) each of
1, 2 and 4 bursts (x4).

In [ ]:
def ratio_table(d):
    out = {}
    for near in ("6", "7"):
        out[f"{near}mm_i_group_means"] = d.center.mean() / d[f"yp{near}"].mean()
        out[f"{near}mm_ii_mean_of_ratios"] = d[f"ratio{near}"].mean()
    return out
ct = m1[(m1.cell_line == "CT26") & (m1.protocol == "cancan") & (m1.rate == "0.2MHz")]
rt = pd.DataFrame({b: ratio_table(ct[ct.bursts == b]) for b in (1, 2, 4, 6)}).T
rt.index.name = "bursts"
print("CT26, 0.2 MHz: center YP over YP near electrodes")
print(rt.round(1).to_string())
print("\nPer-experiment ratios, 1 burst (x4):")
print(ct[ct.bursts == 1][["subject", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(1).to_string(index=False))
inr = (rt >= 36) & (rt <= 55)
print("\nReading A (1 burst) in 36 to 55:", inr.loc[1].to_dict())
print("Reading B (1, 2 and 4 bursts all in 36 to 55):", inr.loc[[1, 2, 4]].all().to_dict())
print("\nBackground readings, ratio for 1 burst (x4) (reading A), and whether 1, 2 and 4 bursts (x4) all lie in 36 to 55 (reading B):")
rows = []
for name, bg in BG_READINGS.items():
    r = {b: ratios_bg(ct[ct.bursts == b].subject, bg.loc[ct[ct.bursts == b].subject]) for b in (1, 2, 4)}
    row = {"background": name, **{f"A_{k}": v for k, v in r[1].items()}}
    row["B_any_reading_all_in_range"] = any(all(36 <= r[b][k] <= 55 for b in (1, 2, 4)) for k in r[1])
    rows.append(row)
print(pd.DataFrame(rows).set_index("background").round(1).to_string())
share, n = assignment_share(list(ct[ct.bursts == 1].subject), 36, 55)
print(f"\nShare of the {n} ways of taking one footprint per well (1 burst (x4)) that put the ratio in 36 to 55:")
print(share.round(3).to_string())


**Result: not reproduced.** With 1 burst (x4) the CT26 ratio at 6 mm is 31.9 (group means) and 34.8
(mean of per-experiment ratios), and at 7 mm 14.9 and 14.4, all below 36. The three experiments give 16.3,
37.4 and 50.7 at 6 mm. Under reading B the 4-burst ratios are 7.7 and 7.9 at 6 mm (2.9 and 6.0 at 7 mm),
far from the range. The result rests on the background: YP at 6 mm is 15 to 48 a.u. after background,
about the spread between footprints. With the lower-left footprint as the background the 1-burst ratio
at 6 mm is 42.3 and 53.7, inside 36 to 55, and with the lower-right one 45.2 (reading ii); over the 64
ways of taking one footprint per well, 62.5% put at least one 6 mm reading in range, while no
footprint brings the 7 mm ratios (12.3 to 17.6) or reading B into range. The outcome is under the
median of the four footprints, an assumption, since the paper does not say which footprint it used.

## fig7-cho-replicates-ct26

> CANCAN remote focusing of electroporation found in CT26 cells also holds in CHO cells, which show a
> central peak of YP uptake that grows with burst number.

For each burst number from 2 bursts (x4) up, the CHO mean center YP must exceed the mean YP at 6 and at
7 mm; together with the CHO part of fig7be-center-linear-with-bursts.

In [ ]:
cho = m1[(m1.cell_line == "CHO") & (m1.protocol == "cancan") & (m1.rate == "0.2MHz")]
g = cho.groupby("bursts")[["center", "yp6", "yp7"]].mean()
g["n"] = cho.groupby("bursts").size()
g["center_over_6"], g["center_over_7"] = g.center / g.yp6, g.center / g.yp7
g["holds"] = (g.center > g.yp6) & (g.center > g.yp7)
print(g.round(2).to_string())
r = lin[("CHO", "a_individual")]
print(f"\nCHO linear fit of center YP on burst number (individual experiments): slope {r['slope']:.1f}, p {r['p']:.2g}")

**Result: reproduced.** For 2, 4 and 6 bursts (x4) the CHO center YP exceeds YP at 6 and 7 mm by the
ratios printed above (1 burst (x4) is reported and also holds), and the CHO center YP rises linearly with
burst number with p < 0.001.

## fig4b-40-fold-at-02mhz

> At a packet rate of 0.2 MHz, YP uptake in the center was 40-fold higher than near the electrodes,
> despite a roughly 4-fold stronger electric field near the electrodes.

CT26, 2 bursts (x4), 0.2 MHz; ratio within 36 to 44 under one of the readings (a) 6 or (b) 7 mm, (i) group
means or (ii) mean of per-experiment ratios.

In [ ]:
two = ct[ct.bursts == 2]
print(two[["subject", "date", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(1).to_string(index=False))
r2 = pd.Series(ratio_table(two))
print("\n", r2.round(2).to_string())
print("\nWithin 36 to 44:", ((r2 >= 36) & (r2 <= 44)).to_dict())
print("\nBackground readings (wells used):")
print(pd.DataFrame({name: ratios_bg(two.subject, bg.loc[two.subject]) for name, bg in BG_READINGS.items()}).T.round(2).to_string())
share, n = assignment_share(list(two.subject), 36, 44)
print(f"\nShare of the {n} ways of taking one footprint per well that put the ratio in 36 to 44:")
print(share.round(3).to_string())
two_alt = pd.concat([two, meas_alt[(meas_alt.spacing == 1.0) & (meas_alt.cell_line == "CT26") & (meas_alt.protocol == "cancan")
                                   & (meas_alt.rate == "0.2MHz") & (meas_alt.bursts == 2)]])
print("\nVariant: the left-out well added:", len(two_alt), "wells")
print(two_alt[["subject", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(1).to_string(index=False))
print(pd.Series(ratio_table(two_alt)).round(2).to_string())
print("Background readings with it:")
print(pd.DataFrame({name: ratios_bg(two_alt.subject, bg.loc[two_alt.subject]) for name, bg in BG_READINGS.items()}).T.round(2).to_string())


**Result: not reproduced.** The ratio at 6 mm is 45.1 of group means and 52.1 as the mean of
per-experiment ratios, just above the 36 to 44 tolerance; at 7 mm it is 10.7 and 12.5. The per-experiment
ratios at 6 mm span 17.9 to 81.9. The dataset has five usable wells here where the paper has n = 3, and
the paper does not say which it used. The field strengths are not in the dataset. The result rests on
the background: with the upper-left footprint the ratio at 6 mm is 35.75 and 38.41, and with the
upper-right one 40.38 (reading i), so single footprints put it in range; over the 1024 ways of taking one
footprint per well, 42.3% put at least one reading in 36 to 44. With the left-out well added (its YP at
6 mm is 207 a.u.) the ratios are 25.57 and 44.23. The outcome is under the median of the four
footprints, an assumption.

## fig3a-cancan-center-enhanced

> All CANCAN protocols enhance electroporation at the center of the array, while the unipolar control
> produces electroporation that decreases toward the center.

CT26, 2 bursts (x4): group-mean ratio of center YP to YP at 6 mm and at 7 mm, above 1 for each CANCAN
rate and below 1 for the control. The dandiset has CANCAN at 1 Hz and 0.2 MHz and controls at 1 Hz only;
the 1 kHz and 10 kHz rates are missing.

In [ ]:
c2 = m1[(m1.cell_line == "CT26") & (m1.bursts == 2)]
g = c2.groupby(["protocol", "rate"])[["center", "yp6", "yp7"]].agg(["mean", "sem"])
g.columns = ["_".join(c) for c in g.columns]
g["n"] = c2.groupby(["protocol", "rate"]).size()
g["center_over_6"] = g.center_mean / g.yp6_mean
g["center_over_7"] = g.center_mean / g.yp7_mean
print(g.round(2).to_string())
print("\nPer control well:")
print(c2[c2.protocol == "control"][["subject", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(2).to_string(index=False))
prof_c = profiles[(profiles.cell_line == "CT26") & (profiles.bursts == 2) & (profiles.protocol == "control")
                  & (profiles.kind == "diagonal") & (profiles.spacing == 1.0)]
gc = prof_c.assign(ax=prof_c.x.abs()).groupby("ax").yp.mean()
print("\nControl, mean YP by absolute distance (a.u.):", gc.round(0).to_dict())
print("Spearman correlation of control YP with distance 0 to 7 mm:", stats.spearmanr(gc.index, gc.values))

**Result: partially reproduced.** Both CANCAN rates present give center-to-edge ratios above 1 (1 Hz:
3.43 at 6 mm and 1.67 at 7 mm; 0.2 MHz: 45.1 and 10.7), but the control ratios are 1.10 and 1.11, where the
criterion needs them below 1. The control YP is nearly flat across the diagonal (2275 a.u. at the center,
1837 to 2085 from 1 to 7 mm; Spearman rho -0.36, p = 0.39), so the decrease toward the center is not
seen. Two of the four CANCAN rates (1 kHz, 10 kHz) are not in the dandiset.

## fig3bc-lorentzian-profile

> The decline of YP uptake sideways from the center is bell-shaped and follows a Lorentzian, with a
> nearly perfect fit within 4 mm of the center (R > 0.9, p < 0.001).

CT26, 2 bursts (x4), CANCAN rates present: Lorentzian fitted to the group-mean profile from -4 to 4 mm,
along the diagonals and along the lines parallel to the sides; R of fitted and observed values in that
range. Shown at ROI spacings of 1 mm (9 points) and 0.5 mm (17 points).

In [ ]:
def group_profile(rate, kind, spacing):
    d = profiles[(profiles.cell_line == "CT26") & (profiles.bursts == 2) & (profiles.protocol == "cancan")
                 & (profiles.rate == rate) & (profiles.kind == kind) & (profiles.spacing == spacing)]
    return d.groupby("x").yp.mean(), d.subject.nunique()
rows, gfits = [], {}
for spacing in (1.0, 0.5):
    for rate in ("1Hz", "0.2MHz"):
        for kind in ("diagonal", "sides"):
            gp, n = group_profile(rate, kind, spacing)
            f = cancan.fit_lorentz(gp.index.values, gp.values, 4)
            gfits[(spacing, rate, kind)] = (gp, f)
            rows.append({"spacing": spacing, "rate": rate, "direction": kind, "n_wells": n, "points": f["n"],
                         "a": f["a"], "fwhm": f["b"], "R": f["R"], "p": f["p"],
                         "p_two_fewer_df": cancan.r_p(f["R"], f["n"], 2), "holds": f["R"] > 0.9 and f["p"] < 0.001})
lor = pd.DataFrame(rows)
print(lor.round(4).to_string(index=False))

**Result: reproduced.** For both rates present and both directions, the Lorentzian fitted from -4 to 4 mm
gives R of 0.93 to 0.99 with p < 0.001 at both ROI spacings (table above). The p takes n - 2 degrees
of freedom; with two fewer, for the two fitted parameters, it stays below 0.001 everywhere except the
1 Hz lines parallel to the sides at 1 mm spacing (p = 0.0027 over 9 points), and all eight pass at
0.5 mm spacing. The 1 kHz and 10 kHz rates are
not in the dandiset.

## fig3bc-offtarget-deviation-falls-with-rate

> Increasing the packet rate from 1 Hz to 0.2 MHz suppresses electroporation near the electrodes and at
> the perimeter between electrodes, so that the excess of YP uptake over the Lorentzian fit at larger
> distances from the center diminishes with packet rate.

Mean of observed minus fitted YP over the ROIs beyond 4 mm, in % of the fitted peak a, for the fits of
fig3bc-lorentzian-profile; smaller at 0.2 MHz than at 1 Hz in both directions.

In [ ]:
rows = []
for (spacing, rate, kind), (gp, f) in gfits.items():
    for limit in ("all", "inside"):
        out = gp[gp.index.map(abs) > 4]
        if limit == "inside":
            # lines parallel to the sides end at the side midpoints, 6.4 mm out
            out = out[out.index.map(abs) <= (cancan.SIDE_MM / 2 if kind == "sides" else 9.05)]
        dev = (out.values - cancan.lorentz(out.index.values, f["a"], f["b"])).mean() / f["a"] * 100
        rows.append({"spacing": spacing, "direction": kind, "rois": limit, "rate": rate, "points_beyond_4mm": len(out), "deviation_pct_of_a": dev})
dv = pd.DataFrame(rows).pivot_table(index=["rois", "spacing", "direction"], columns="rate", values=["deviation_pct_of_a", "points_beyond_4mm"])
dv[("smaller_at_0.2MHz", "")] = dv[("deviation_pct_of_a", "0.2MHz")] < dv[("deviation_pct_of_a", "1Hz")]
print(dv.round(2).to_string())


**Result: reproduced.** Beyond 4 mm the YP excess over the Lorentzian fit is 1.7 to 2.7% of the fitted
peak at 0.2 MHz and 24 to 30% at 1 Hz, along the diagonals and along the lines parallel to the sides, at
both ROI spacings. The lines parallel to the sides end at the side midpoints, 6.4 mm out; keeping only
the ROIs inside that gives 2.0 and 2.3% at 0.2 MHz against 27.5 and 27.3% at 1 Hz. Only these two rates
are in the dandiset.

## fig4b-center-no-rate-effect

> YP uptake in the center of the array shows no significant effect of the packet rate.

A claim of absence. CT26, 2 bursts (x4), center YP at each rate against 1 Hz by two-sided Student's
t-tests with Bonferroni correction. Only 0.2 MHz can be compared with 1 Hz here; the correction is shown
for the one comparison available and for the paper's three (1 kHz, 10 kHz and 0.2 MHz against 1 Hz).

In [ ]:
c_hz = c2[(c2.protocol == "cancan") & (c2.rate == "1Hz")].center
c_mhz = c2[(c2.protocol == "cancan") & (c2.rate == "0.2MHz")].center
t = stats.ttest_ind(c_mhz, c_hz)
w = stats.ttest_ind(c_mhz, c_hz, equal_var=False)
print(f"1 Hz: mean {c_hz.mean():.0f} +/- {c_hz.sem():.0f} (n {len(c_hz)}); 0.2 MHz: mean {c_mhz.mean():.0f} +/- {c_mhz.sem():.0f} (n {len(c_mhz)})")
print(f"difference {c_mhz.mean() - c_hz.mean():.0f} a.u. ({(c_mhz.mean() / c_hz.mean() - 1) * 100:.0f}%); "
      f"Student t = {t.statistic:.2f}, p = {t.pvalue:.3f}; Bonferroni x1 p = {min(1, t.pvalue):.3f}, x3 p = {min(1, 3 * t.pvalue):.3f}")
print(f"Welch variant: p = {w.pvalue:.3f}")

**Result: reproduced.** The center YP at 0.2 MHz is 36% lower than at 1 Hz (1985 against 3088 a.u.),
and the t-test does not find the difference (p = 0.257 uncorrected, 0.770 with the paper's threefold
correction). With two 1 Hz wells the test has little power, and the 1 kHz and 10 kHz comparisons cannot
be made.

## fig4b-near-electrode-rate-decline

> YP uptake at 6 and 7 mm from the center (near the electrodes) declines steeply with increasing packet
> rate, so that higher rates suppress off-target electroporation despite a stronger electric field there.

CT26, 2 bursts (x4): power function of packet rate fitted to YP at 6 and at 7 mm, to group means and to
individual experiments (two rates only: 1 Hz and 0.2 MHz), and 0.2 MHz against 1 Hz by a two-sided t-test.

In [ ]:
cc = c2[c2.protocol == "cancan"].assign(rate_hz=lambda d: d.rate.map({"1Hz": 1.0, "0.2MHz": 208333.0}))
for near in ("yp6", "yp7"):
    g = cc.groupby("rate_hz")[near].mean()
    k_group = np.log(g.iloc[1] / g.iloc[0]) / np.log(g.index[1] / g.index[0])
    (c_, k_ind), _ = optimize.curve_fit(lambda x, c, k: c * x ** k, cc.rate_hz, cc[near], p0=[g.iloc[0], -0.2], maxfev=20000)
    t = stats.ttest_ind(cc[cc.rate_hz > 1][near], cc[cc.rate_hz == 1][near])
    print(f"{near}: 1 Hz {g.iloc[0]:.0f}, 0.2 MHz {g.iloc[1]:.0f} a.u. (ratio {g.iloc[1] / g.iloc[0]:.3f}); exponent, group means {k_group:.3f}, "
          f"individual experiments {k_ind:.3f}; t = {t.statistic:.2f}, p = {t.pvalue:.2g}")

**Result: reproduced.** YP at 0.2 MHz is 4.9% of YP at 1 Hz at 6 mm and 10.0% at 7 mm; the
power-function exponents are -0.246 and -0.188 under both fits, and the t-tests give p = 3.2e-06 and
0.00036. Only two rates are in the dandiset, so the power function passes through two group means.

## fig4c-fwhm-decreases-with-rate

> Higher packet rates focus electroporation into a smaller spot, FWHM decreasing from about 4 mm at 1 Hz
> to about 2 mm at 1 kHz, 10 kHz and 0.2 MHz, less than 15% of the gap between electrodes.

CT26, 2 bursts (x4); per-experiment FWHM fitted from -4 to 4 mm; ROI spacings 1 and 0.5 mm. Only 1 Hz and
0.2 MHz are in the dandiset.

In [ ]:
rows = []
for spacing in (1.0, 0.5):
    d = meas[(meas.cell_line == "CT26") & (meas.bursts == 2) & (meas.protocol == "cancan") & (meas.spacing == spacing)]
    hz, mhz = d[d.rate == "1Hz"].fwhm4, d[d.rate == "0.2MHz"].fwhm4
    t = stats.ttest_ind(mhz, hz)
    rows.append({"spacing": spacing, "fwhm_1Hz": hz.mean(), "sem_1Hz": hz.sem(), "n_1Hz": len(hz),
                 "fwhm_0.2MHz": mhz.mean(), "sem_0.2MHz": mhz.sem(), "n_0.2MHz": len(mhz),
                 "p": t.pvalue, "p_bonferroni_x3": min(1, 3 * t.pvalue),
                 "1Hz_in_3.6_4.4": abs(hz.mean() - 4) <= 0.4, "0.2MHz_in_1.8_2.2": abs(mhz.mean() - 2) <= 0.2,
                 "min_pct_of_gap": min(hz.mean(), mhz.mean()) / 15.3 * 100})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nPer-experiment FWHM at 1 Hz (spacing 1 mm, fit ranges 4, 5, 6 mm):")
print(m1[(m1.cell_line == "CT26") & (m1.rate == "1Hz") & (m1.protocol == "cancan")][["subject", "fwhm4", "fwhm5", "fwhm6"]].round(2).to_string(index=False))

**Result: partially reproduced.** The FWHM falls from 4.56 mm at 1 Hz to 2.00 mm at 0.2 MHz (1 mm
spacing; 4.54 and 1.88 mm at 0.5 mm spacing). The 0.2 MHz mean is within 1.8 to 2.2 mm and is 12 to 13%
of the gap, below 15%, but the 1 Hz mean is above the 3.6 to 4.4 mm tolerance (two wells, 3.91 and
5.21 mm). The 1 kHz and 10 kHz means and their tests, which the criterion needs, cannot be computed. The
0.2 MHz against 1 Hz difference has p = 0.009 with the threefold correction, where the paper reports 0.09.

### Variant for the CT26 2 bursts (x4) rate claims: the left-out well added

Well P1_20240627_B1 (CT26, 0.2 MHz, 2 bursts (x4)) was left out by the placement rule. Here it is added,
placed with the angle of its plate and day, and the numbers of fig3a-cancan-center-enhanced,
fig4b-center-no-rate-effect, fig4b-near-electrode-rate-decline and fig4c-fwhm-decreases-with-rate are
recomputed.

In [ ]:
c2a = pd.concat([c2, meas_alt[(meas_alt.spacing == 1.0) & (meas_alt.cell_line == "CT26") & (meas_alt.bursts == 2)]])
ca = c2a[c2a.protocol == "cancan"]
mhz, hz = ca[ca.rate == "0.2MHz"], ca[ca.rate == "1Hz"]
print(f"0.2 MHz wells: {len(mhz)}")
print(f"fig3a: 0.2 MHz center over 6 mm {mhz.center.mean() / mhz.yp6.mean():.2f}, over 7 mm {mhz.center.mean() / mhz.yp7.mean():.2f}")
t = stats.ttest_ind(mhz.center, hz.center)
print(f"fig4b center: 0.2 MHz {mhz.center.mean():.0f} against 1 Hz {hz.center.mean():.0f}; p {t.pvalue:.3f}, x3 {min(1, 3 * t.pvalue):.3f}")
for k in ("yp6", "yp7"):
    t = stats.ttest_ind(mhz[k], hz[k])
    expo = np.log(mhz[k].mean() / hz[k].mean()) / np.log(208333.0)
    print(f"fig4b near electrodes {k}: 0.2 MHz {mhz[k].mean():.0f} against 1 Hz {hz[k].mean():.0f}; exponent (group means) {expo:.3f}; p {t.pvalue:.2g}")
ma = meas_alt[(meas_alt.cell_line == "CT26") & (meas_alt.bursts == 2) & (meas_alt.rate == "0.2MHz")]
for spacing in (1.0, 0.5):
    d = pd.concat([meas[(meas.cell_line == "CT26") & (meas.bursts == 2) & (meas.protocol == "cancan") & (meas.rate == "0.2MHz") & (meas.spacing == spacing)],
                   ma[ma.spacing == spacing]])
    print(f"fig4c spacing {spacing}: 0.2 MHz FWHM {d.fwhm4.mean():.2f} mm (s.e. {d.fwhm4.sem():.2f}, n {len(d)}); left-out well {ma[ma.spacing == spacing].fwhm4.iloc[0]:.2f} mm")


With the left-out well added (6 wells at 0.2 MHz) the conclusions of these four claims do not change: the
0.2 MHz center-to-edge ratios stay above 1 (25.57 and 10.76); center YP at 0.2 MHz against 1 Hz has
p = 0.175 (0.525 corrected); YP near the electrodes still falls (exponents -0.207 and -0.195, p 9.1e-06
and 7.6e-05); and the 0.2 MHz FWHM is 2.28 mm at 1 mm spacing (outside 1.8 to 2.2) and 2.17 mm at 0.5 mm
spacing, while the 1 Hz FWHM, unchanged, misses its tolerance either way.


## fig5b-pr-gaussian-yp-lorentzian (blocked; YP half shown)

> The Pr (cell killing) profile is better fitted by a Gaussian than by a Lorentzian (R2 0.96 against
> 0.92), while the YP profile is better fitted by a Lorentzian (0.98 against 0.95).

The dandiset holds no propidium images, so the claim is blocked. The YP half can be computed on the CT26
0.2 MHz 2 bursts (x4) wells, which match the 0 mm elevation condition of Fig. 5: group-mean diagonal
profile in % of its center value, Lorentzian and Gaussian fitted from -5 to 5 mm (Results 3.2), R2 of
each. Expected 0.98 and 0.95 (tolerance 10%), with the Lorentzian better.

In [ ]:
rows = []
for spacing in (1.0, 0.5):
    gp, n = group_profile("0.2MHz", "diagonal", spacing)
    pct = gp / gp.loc[0.0] * 100
    fl, fg = cancan.fit_lorentz(pct.index.values, pct.values, 5), cancan.fit_gauss(pct.index.values, pct.values, 5)
    rows.append({"spacing": spacing, "n_wells": n, "points": fl["n"], "R2_lorentzian": fl["R2"], "R2_gaussian": fg["R2"],
                 "fwhm_lorentzian": fl["b"], "fwhm_gaussian": fg["fwhm"], "lorentzian_better": fl["R2"] > fg["R2"],
                 "lorentzian_in_tolerance": abs(fl["R2"] - 0.98) <= 0.098, "gaussian_in_tolerance": abs(fg["R2"] - 0.95) <= 0.095})
print(pd.DataFrame(rows).round(4).to_string(index=False))


The YP half holds: R2 is 0.979 for the Lorentzian and 0.975 for the Gaussian at 1 mm spacing (0.982 and
0.979 at 0.5 mm), both within tolerance and with the Lorentzian better, though by little. The claim stays
blocked, since its Pr half cannot be tested.


## fig5c-irreversible-center (blocked; YP part and pre-exposure Hoechst shown)

The criterion asks that center YP exceed YP at every distance from 2 to 7 mm and that Hoechst inside the
electrode footprint be above background. The paper's Hoechst check was made after exposure: the Pr
samples "were also re-imaged for Hoechst fluorescence, to check for possible cell loss due to detachment"
(Methods 2.4), and Hoechst "proved that cells were still present and the lack of YP staining was not a
result of cell detachment" (Results 3.2). The dandiset's Hoechst images were taken before exposure, so they
cannot show that cells stayed; and the paper defines no Hoechst background. Both parts are shown on the
CT26 0.2 MHz 2 bursts (x4) wells for reference: YP by distance (ROI spacing 1 mm), and pre-exposure
Hoechst inside each footprint (the 3 background ROIs) against the monolayer (diagonal ROIs within 5 mm of
the center) and the camera offset (median of the empty second planes found above).


In [ ]:
pr = profiles[(profiles.cell_line == "CT26") & (profiles.bursts == 2) & (profiles.rate == "0.2MHz")
              & (profiles.protocol == "cancan") & (profiles.kind == "diagonal") & (profiles.spacing == 1.0)]
gm = pr.assign(ax=pr.x.abs()).groupby("ax").yp.mean()
print("Mean YP by absolute distance (a.u.):", gm.round(0).to_dict())
print("Center above every distance from 2 to 7 mm:", bool((gm.loc[0] > gm.loc[2:7]).all()))
sel = wells[(wells.cell_line == "CT26") & (wells.protocol == "cancan") & (wells.rate == "0.2MHz") & (wells.bursts == 2)].index
offset = odd_planes[odd_planes.plane == 1]["median"].median()
rows = []
for _, r in meta[(meta.channel == "DAPI") & meta.subject.isin(sel)].iterrows():
    h = cancan.load_binned(r.url)["image"]
    sq = squares[r.subject] if fits.loc[r.subject, "used"] else alt_squares[r.subject]
    fpb = cancan.background(h, sq)["per_footprint"]
    p = cancan.profile(h, sq, cancan.distances(1.0), "diagonal")
    mono = p[p.x_mm.abs() <= 5].raw.median()
    rows.append({"subject": r.subject, "used": fits.loc[r.subject, "used"], "footprint_min": np.min(fpb), "footprint_max": np.max(fpb),
                 "monolayer_median": mono, "footprint_min_over_monolayer": np.min(fpb) / mono, "footprint_min_over_offset": np.min(fpb) / offset})
print(f"\nPre-exposure Hoechst (a.u.); camera offset {offset:.1f}")
print(pd.DataFrame(rows).round(2).to_string(index=False))


The YP part holds (center 1985 a.u. against at most 244 from 2 to 7 mm). Before exposure, Hoechst inside
every footprint is 0.74 to 0.99 times the monolayer level and 20 to 60 times the camera offset, so cells
were present there before exposure; whether they stayed, which is what the paper's post-exposure Hoechst
showed, cannot be seen in this dandiset. The claim stays blocked.


## Summary

| Claim | Outcome |
|---|---|
| fig7be-center-linear-with-bursts | partially reproduced (assumption: wells left out) |
| fig7b-ct26-saturation | partially reproduced |
| fig7cf-fwhm-with-bursts | reproduced |
| fig7-sharpest-peak-one-burst | reproduced (assumption: 0.5 mm ROI spacing) |
| fig7-offtarget-40-50-fold | not reproduced (assumption: background, median of four footprints) |
| fig7-cho-replicates-ct26 | reproduced |
| fig4b-40-fold-at-02mhz | not reproduced (assumption: background, median of four footprints) |
| fig3a-cancan-center-enhanced | partially reproduced |
| fig3bc-lorentzian-profile | reproduced |
| fig3bc-offtarget-deviation-falls-with-rate | reproduced |
| fig4b-center-no-rate-effect | reproduced |
| fig4b-near-electrode-rate-decline | reproduced |
| fig4c-fwhm-decreases-with-rate | partially reproduced |
| fig4c-log-dependence | blocked (other: not in this dandiset, a rate series is in DANDI:001047) |
| fig5-raised-electrodes-weaker-center | blocked (other: in DANDI:001284) |
| fig5b-pr-gaussian-yp-lorentzian | blocked (other: in DANDI:001284; the YP half holds) |
| fig5-yp-spot-slightly-larger-than-pr | blocked (other: in DANDI:001284) |
| fig5c-irreversible-center | blocked (other: post-exposure Hoechst in DANDI:001284; the YP part holds) |
| fig6b-900ns-peak-not-significant | blocked (other: in DANDI:001277) |
| fig6c-normalized-decline-similar | blocked (other: in DANDI:001277) |
| fig6d-fwhm-sharper-longer-pulses | blocked (other: in DANDI:001277) |
| fig2-heating-below-44c | blocked (data missing) |
| fig1e-field-simulation | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** ROIs are placed by a program, not by hand in MetaMorph:
the array center is the center of a square of the paper's size fitted to the dark electrode footprints,
on images reduced to 16.25 um bins, with the pixel size of 1.625 um that the OME-XML in the descriptions
gives. Three wells whose footprints could not be located were left out; with them added, placed with the
angle of their plate and day, the CT26 slope of fig7be-center-linear-with-bursts reaches p < 0.001. The
background is the median over the four footprints of 3 ROIs each, where the paper uses 3 ROIs in one
footprint without saying which; the two ratio claims (fig7-offtarget-40-50-fold, fig4b-40-fold-at-02mhz)
fall in range under some single footprints, so their outcomes rest on this choice. The ROI spacing and the fit
range are not given in the paper and are run under each reading; where they change an outcome
(fig7-sharpest-peak-one-burst) the 0.5 mm spacing is adopted as an assumption. The wells are those deposited, which
for most groups are more than the paper's n, and the paper does not say which it used.

**What was found about the dataset.** The dandiset holds the Fig. 7 burst series and 1 Hz CANCAN and
control wells, but none of the other series of the paper, which three other dandisets citing the paper
appear to hold (DANDI:001047, 001277 and 001284); 001276 does not point to them. Electrode positions and the
experimental conditions are not in structured metadata (conditions only in free text, pixel size and
exposure only in OME-XML appended to the free-text description);
`session_start_time` is the conversion date; two Hoechst files carry an undocumented second plane and one
well has a duplicated Hoechst fragment, another has no Hoechst file. The well counts exceed the paper's.